# Image Captioning with CNN Encoder + LSTM Decoder
### CSC 483 - Assignment

---

## Learning Objectives

By the end of this assignment you will be able to:

1. **Preprocess** raw image-caption data into clean, tokenized sequences suitable for training.
2. **Extract visual features** from images using a frozen pretrained CNN (InceptionV3).
3. **Build a dual-input Keras model** that merges image features with text through an attention mechanism and decodes captions with an LSTM.
4. **Train** the model using teacher forcing, monitor loss with callbacks, and interpret learning curves.
5. **Generate captions** at inference time using greedy decoding.
6. **Evaluate** caption quality quantitatively with BLEU scores and qualitatively by inspecting outputs.


## Background

**Image captioning** is the task of automatically generating a natural-language description of an image. It sits at the intersection of computer vision and natural language processing and is a core benchmark for multimodal understanding.

The standard neural approach uses an **encoder-decoder** architecture:
- A pretrained **CNN encoder** (here, InceptionV3) extracts spatial feature maps from the image.
- An **LSTM decoder** generates the caption one word at a time, attending to different image regions at each step.

During training we use **teacher forcing** — the decoder receives the ground-truth previous word at each step. At inference time there is no ground truth, so we switch to **greedy** (or beam-search) decoding.

### How InceptionV3 is adapted for this task

InceptionV3 is pretrained on ImageNet for 1000-class classification. We repurpose it as a **frozen spatial feature extractor** by changing three settings:

| Setting | Value used | Why |
|---------|-----------|-----|
| `include_top=False` | Removes classification head | We need feature maps, not class scores |
| `pooling=None` | No global pooling | Preserves the spatial H×W grid — required for attention |
| `trainable=False` | Frozen weights | Avoids overwriting ImageNet knowledge; saves GPU memory |

With these settings the encoder outputs shape **(batch, H, W, 2048)**, which we flatten to **(batch, N_PATCHES, 2048)** — one 2048-dim descriptor per spatial patch. The attention mechanism then learns a different weighting over these patches for each word it generates.

The decoder (Embedding → LSTM → attention → LSTM → Dense) is trained from scratch on Flickr8k. This is the **non-trivial training stage** — we are learning how to combine image regions with language to generate captions.

## Dataset — Flickr8k

We use the [Flickr8k](https://www.kaggle.com/datasets/adityajn105/flickr8k) dataset:

- **~8,000 images** of everyday scenes (people, animals, outdoor activities).
- **5 human-written captions per image** (40,000 captions total).
- A single CSV file (`captions.txt`) mapping each `image,caption` pair.

The dataset is small enough to train on a free-tier Colab GPU within.


## Grading Rubric

| Part | Task | Points |
|------|------|-------:|
| 1 | Load Captions | 3 |
| 1 | Clean Captions | 5 |
| 1 | Build Vocabulary | 5 |
| 1 | Mappings & Tokenization | 7 |
| 2 | Load InceptionV3 | 5 |
| 2 | Extract & Cache Features | 10 |
| 3 | Build Attention Model | 15 |
| 3 | Data Generator + Split | 10 |
| 4 | Training Loop | 15 |
| 5 | Greedy Caption Generation | 15 |
| 6 | BLEU Evaluation + Qualitative Analysis | 10 |
| **Total** | | **100** |
| Bonus | Beam Search | +10 |

> **Important:** Your notebook must run end-to-end (`Kernel → Restart and Run All`) on a Colab GPU. All code must use **Keras** (TensorFlow backend). PyTorch is **not** permitted.

> **Note on caption quality:** Your BLEU score and the visual quality of generated captions  
> **do not affect your grade** as long as your implementation is logically correct.  
> A working pipeline with low BLEU (e.g. due to limited training time or compute) will still  
> receive full marks we are grading your **code and understanding**, not the model's output quality.

## Setup

> **If running on Colab, ensure your runtime is set to GPU before starting.**  
> Go to **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**, then click **Save**.  
> Feature extraction and training will be extremely slow on CPU.

Run this cell to install dependencies and import all libraries you will need. **Do not modify.**

In [ ]:
# ─── PROVIDED — do not modify ───
!pip install nltk kagglehub -q

import os, re, pickle, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import Counter

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

print("TF:", tf.__version__)
print("GPU:", len(tf.config.list_physical_devices('GPU')) > 0)


## Download Dataset

Run this cell to download Flickr8k via Kaggle. **Do not modify.**

In [ ]:
# ─── PROVIDED — do not modify ───
import kagglehub
dataset_path = kagglehub.dataset_download("adityajn105/flickr8k")
print("Dataset downloaded to:", dataset_path)

IMAGES_DIR    = os.path.join(dataset_path, "Images")
CAPTIONS_FILE = os.path.join(dataset_path, "captions.txt")

assert os.path.isdir(IMAGES_DIR), f"Images directory not found at {IMAGES_DIR}"
assert os.path.isfile(CAPTIONS_FILE), f"Captions file not found at {CAPTIONS_FILE}"
print(f"Images: {len(os.listdir(IMAGES_DIR))} files")
print(f"Captions file: {CAPTIONS_FILE}")


---
## Part 1 — Text Preprocessing (20 pts)

In this section you will load the raw caption data, clean it, build a vocabulary, and convert captions to integer sequences.


### 1.1 Load Captions (3 pts)

Write a function `load_captions(filepath)` that reads `captions.txt` and returns a dictionary mapping each image filename to a **list** of its caption strings.

**Requirements:**
- Skip the header row (first line of the file).
- Split each line on the **first comma only** — use `split(',', 1)` so commas inside the caption text are preserved.
- Strip whitespace from both the image name and the caption.
- Each image has exactly 5 captions.

**Expected output:**
```
Total images: 8092
  A child in a pink dress is climbing up a set of stairs in an entry way .
  A girl going into a wooden building .
  A little girl climbing into a wooden playhouse .
  A little girl climbing the stairs to her playhouse .
  A little girl in a pink dress going into a wooden cabin .
```

In [ ]:
def load_captions(filepath):
    captions = {}
    # TODO: open the file, skip the header line, then for each line:
    #   1. Strip whitespace
    #   2. Split on the FIRST comma only
    #   3. Append the caption to the list for that image name

    # ---- YOUR CODE HERE ----
    with open(filepath, 'r', encoding='utf-8') as f:
        next(f)  # skip header
        for line in f:
            line = line.strip()
            if not line:
                continue
            img_name, caption = line.split(',', 1)
            img_name = img_name.strip()
            caption = caption.strip()
            captions.setdefault(img_name, []).append(caption)
    # ---- END YOUR CODE ----
    return captions

captions_dict = load_captions(CAPTIONS_FILE)
sample_img = list(captions_dict.keys())[0]
print(f"Total images: {len(captions_dict)}")
for c in captions_dict[sample_img]:
    print(" ", c)


### 1.2 Clean Captions (5 pts)

Write `clean_caption(caption)` that applies **all four** of the following steps **in order**:
1. Lowercase the string.
2. Remove all non-alphabetic characters (keep spaces) — hint: `re.sub(r"[^a-z\s]", "", caption)`.
3. Remove single-character words **except** `'a'`.
4. Strip leading/trailing whitespace.

`clean_all_captions(captions_dict)` is provided — it applies `clean_caption` to every caption in the dictionary. You only need to implement `clean_caption`.

**Expected output:**
```
Before: A child in a pink dress is climbing up a set of stairs in an entry way .
After:  child in pink dress is climbing up set of stairs in an entry way
```

*(Lowercased, punctuation removed, extra whitespace collapsed, leading/trailing whitespace stripped.)*

In [ ]:
def clean_caption(caption):
    # TODO: implement the four cleaning steps described above
    #   1. caption.lower()
    #   2. re.sub(r"[^a-z\s]", "", caption)
    #   3. ' '.join(w for w in caption.split() if len(w) > 1 or w == 'a')
    #   4. caption.strip()

    # ---- YOUR CODE HERE ----
    caption = caption.lower()
    caption = re.sub(r"[^a-z\s]", "", caption)
    caption = ' '.join(w for w in caption.split() if len(w) > 1 or w == 'a')
    caption = caption.strip()
    # ---- END YOUR CODE ----
    return caption


# PROVIDED — do not modify
def clean_all_captions(captions_dict):
    return {img: [clean_caption(c) for c in caps]
            for img, caps in captions_dict.items()}


captions_clean = clean_all_captions(captions_dict)
print("Before:", captions_dict[sample_img][0])
print("After: ", captions_clean[sample_img][0])


### 1.3 Build Vocabulary (5 pts)

Write `build_vocabulary(captions_dict, min_freq=5)` that:
1. Counts the frequency of every word across all cleaned captions.
2. Returns a **set** of words that appear at least `min_freq` times.

**Question:** Why do we filter out rare words? What is the trade-off? *(Answer in a comment or print statement.)*

**Expected output:** `Vocab size:` roughly 2,904 – 3,104 (vocab words + 4 special tokens)

*(Expect a value in the range 2,900 – 3,100 depending on your cleaning.)*


In [ ]:
def build_vocabulary(captions_dict, min_freq=5):
    # TODO:
    #   1. Create a Counter over all words in all captions
    #   2. Return the set of words whose count >= min_freq

    # ---- YOUR CODE HERE ----
    counter = Counter()
    for caps in captions_dict.values():
        for cap in caps:
            counter.update(cap.split())

    vocab_set = {word for word, count in counter.items() if count >= min_freq}
    return vocab_set
    # ---- END YOUR CODE ----

vocab = build_vocabulary(captions_clean, min_freq=5)
print(f"Vocab size: {len(vocab)}")   # Expected: 2904 – 3104

# TODO: Answer — why filter rare words? What's the trade-off?
# Your answer: Filtering rare words keeps the vocabulary smaller, reduces sparsity,
# memory use, and overfitting, and makes training/inference more stable. The trade-off
# is that uncommon but meaningful words are mapped to <unk>, so captions may lose detail.


### 1.4 Mappings & Tokenization (7 pts)

Implement three functions:

1. **`build_mappings(vocab)`** — Create `word2idx` and `idx2word` dictionaries. Reserve indices 0–3 for special tokens at these **exact** positions:

   | Token | Index |
   |-------|-------|
   | `<pad>` | 0 |
   | `<start>` | 1 |
   | `<end>` | 2 |
   | `<unk>` | 3 |

   Then assign indices 4+ to the **sorted** vocabulary words (sorted order ensures reproducibility).

2. **`add_tokens(captions_dict)`** — Wrap every caption string with `<start>` and `<end>` tokens.

3. **`captions_to_sequences(captions_dict, word2idx)`** — Convert each token to its integer index. Unknown words must map to `word2idx['<unk>']` — use `.get(w, unk_idx)`, not `word2idx[w]` (which would crash on OOV words).

After running these, compute `MAX_LEN` (length of the longest sequence) and `VOCAB_SIZE`.

**Expected output:**
```
MAX_LEN: 35 – 40   |   VOCAB_SIZE: 2904 – 3104
Special token indices: {'<pad>': 0, '<start>': 1, '<end>': 2, '<unk>': 3}
```

*(MAX_LEN should be 35–40. VOCAB_SIZE = vocab + 4 special tokens. Sequences should start with `1` (`<start>`) and end with `2` (`<end>`).)*

In [ ]:
def build_mappings(vocab):
    # TODO: create word2idx with special tokens at indices 0-3,
    #       then sorted vocab words starting at index 4.
    #       Also create the reverse mapping idx2word.

    # ---- YOUR CODE HERE ----
    word2idx = {
        '<pad>': 0,
        '<start>': 1,
        '<end>': 2,
        '<unk>': 3
    }

    for i, word in enumerate(sorted(vocab), start=4):
        word2idx[word] = i

    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word
    # ---- END YOUR CODE ----


def add_tokens(captions_dict):
    # TODO: prepend <start> and append <end> to each caption string
    #       return a dict with the same structure

    # ---- YOUR CODE HERE ----
    return {
        img: [f"<start> {cap} <end>" for cap in caps]
        for img, caps in captions_dict.items()
    }
    # ---- END YOUR CODE ----


def captions_to_sequences(captions_dict, word2idx):
    # TODO: convert each word to its integer index.
    #       Use word2idx.get(w, word2idx['<unk>']) for out-of-vocabulary words.

    # ---- YOUR CODE HERE ----
    unk_idx = word2idx['<unk>']
    return {
        img: [[word2idx.get(w, unk_idx) for w in cap.split()] for cap in caps]
        for img, caps in captions_dict.items()
    }
    # ---- END YOUR CODE ----


word2idx, idx2word = build_mappings(vocab)
captions_tokens    = add_tokens(captions_clean)
captions_seqs      = captions_to_sequences(captions_tokens, word2idx)

all_lengths = [len(seq) for seqs in captions_seqs.values() for seq in seqs]
MAX_LEN     = max(all_lengths)
VOCAB_SIZE  = len(word2idx)
print(f"MAX_LEN: {MAX_LEN} | VOCAB_SIZE: {VOCAB_SIZE}")
# Expected: MAX_LEN 35-40, VOCAB_SIZE 2904-3104
print("Special token indices:", {w: word2idx[w] for w in ['<pad>','<start>','<end>','<unk>']})
# Expected: {'<pad>': 0, '<start>': 1, '<end>': 2, '<unk>': 3}
print("Seq sample:", captions_seqs[sample_img][0])


---
## Part 2 — Feature Extraction (15 pts)

We use a frozen **InceptionV3** (pretrained on ImageNet) to extract spatial feature maps from every image. With `pooling=None` and `include_top=False`, the output is an `H×W×2048` feature grid (typically 8×8 or 7×7 depending on TF version), which we reshape to `(N_PATCHES, 2048)` — one 2048-dimensional descriptor per spatial patch. The attention mechanism in Part 3 will learn to focus on different patches for different words.


### 2.1 Load InceptionV3 (5 pts)

Write `build_feature_extractor()` that:
1. Loads InceptionV3 with ImageNet weights, `include_top=False`, input shape `(299, 299, 3)`, and **`pooling=None`**.
2. Sets `base.trainable = False` to freeze all weights.
3. Returns the model.

>  **`pooling=None` is critical.** Using `pooling='avg'` collapses the spatial grid to a single `(2048,)` vector and breaks the attention mechanism in Part 3.

**Expected output:**
```
Output shape: (None, 8, 8, 2048) or (None, 7, 7, 2048)  ← depends on TF version
Grid: 8x8 = 64 patches  OR  7x7 = 49 patches
Trainable weights: 0
```

In [ ]:
def build_feature_extractor():
    # TODO: load InceptionV3 with the settings described above

    # ---- YOUR CODE HERE ----
    base = InceptionV3(
        weights='imagenet',
        include_top=False,
        input_shape=(299, 299, 3),
        pooling=None
    )
    base.trainable = False
    return base
    # ---- END YOUR CODE ----

extractor = build_feature_extractor()
out_shape = extractor.output_shape
print("Output shape:", out_shape)          # Expected: (None, 8, 8, 2048) or (None, 7, 7, 2048)

SPATIAL_H = out_shape[1]
SPATIAL_W = out_shape[2]
N_PATCHES = SPATIAL_H * SPATIAL_W
print(f"Grid: {SPATIAL_H}x{SPATIAL_W} = {N_PATCHES} patches")
assert out_shape[3] == 2048, "Feature depth should be 2048"
print("Trainable weights:", len(extractor.trainable_weights))  # Expected: 0


### 2.2 Extract & Cache Features (10 pts)

Implement two functions:

1. **`preprocess_image(img_path, target_size=(299,299))`** — Open an image with PIL, resize to `target_size`, convert to a `float32` numpy array, add a batch axis with `np.expand_dims`, then apply `preprocess_input` (InceptionV3-specific normalisation: pixels → [−1, 1]).

2. **`extract_all_features(images_dir, extractor, save_path, batch_size=32)`** — Loop through all `.jpg` files in batches. For each batch: preprocess images, run `extractor.predict(...)`, **reshape** the output from `(B, H, W, 2048)` to `(B, N_PATCHES, 2048)`, and store each image's feature array in a dictionary keyed by filename. Save the dictionary to disk with `pickle.dump`.

The `load_or_extract` wrapper (provided below) checks for a cached `.pkl` file first — features only need to be extracted once (~5–10 min on a Colab GPU).

#### Expected Output (2.2)
```
Loaded 8091 cached features, shape=(64, 2048)
Feature shape per image: (64, 2048)
```
*(Shape should be `(64, 2048)` or `(49, 2048)` depending on TF version.)*

In [ ]:
def preprocess_image(img_path, target_size=(299, 299)):
    # TODO:
    #   1. Open image with PIL, convert to RGB
    #   2. Resize to target_size
    #   3. Convert to float32 numpy array
    #   4. Expand dims (add batch axis)
    #   5. Apply preprocess_input

    # ---- YOUR CODE HERE ----
    img = Image.open(img_path).convert('RGB')
    img = img.resize(target_size)
    arr = np.array(img, dtype=np.float32)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)
    return arr
    # ---- END YOUR CODE ----

def extract_all_features(images_dir, extractor, save_path, batch_size=32):
    features    = {}
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]

    for batch_start in tqdm(range(0, len(image_files), batch_size), desc="Extracting"):
        batch_files = image_files[batch_start : batch_start + batch_size]
        batch_arrs, valid_names = [], []
        for img_name in batch_files:
            try:
                arr = preprocess_image(os.path.join(images_dir, img_name))
                batch_arrs.append(arr[0])
                valid_names.append(img_name)
            except Exception as e:
                print(f"Skipping {img_name}: {e}")

        if not batch_arrs:
            continue

        batch_np = np.stack(batch_arrs, axis=0)
        # TODO: run extractor.predict on the batch, then reshape
        #       from (B, 8, 8, 2048) to (B, N_PATCHES, 2048)

        # ---- YOUR CODE HERE ----
        batch_feats = extractor.predict(batch_np, verbose=0)
        batch_feats = batch_feats.reshape(batch_feats.shape[0], -1, batch_feats.shape[-1])

        for img_name, feat in zip(valid_names, batch_feats):
            features[img_name] = feat.astype(np.float32)
        # ---- END YOUR CODE ----

    # TODO: save features dict to save_path using pickle

    # ---- YOUR CODE HERE ----
    with open(save_path, 'wb') as f:
        pickle.dump(features, f)
    # ---- END YOUR CODE ----
    print(f"Saved {len(features)} features to {save_path}")
    return features

# ─── PROVIDED — caching wrapper ───
FEATURES_PATH = "features_attention.pkl"

def load_or_extract(path, images_dir, extractor):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            cached = pickle.load(f)
        if cached:
            sample_shape = next(iter(cached.values())).shape
            if sample_shape == (N_PATCHES, 2048):
                print(f"Loaded {len(cached)} cached features, shape={sample_shape}")
                return cached
            print(f"Cache shape mismatch {sample_shape} vs expected {(N_PATCHES, 2048)}; re-extracting.")
    return extract_all_features(images_dir, extractor, path)

features = load_or_extract(FEATURES_PATH, IMAGES_DIR, extractor)
print("Feature sample shape:", features[sample_img].shape)   # Expected: (N_PATCHES, 2048)


---
## Part 3 — Model Architecture (25 pts)

You will build a dual-input Keras model that combines image patches with partial captions through a **dot-product attention** mechanism, then decodes the next word with an LSTM.


### What is Attention?

**Attention** is a mechanism that allows the decoder to dynamically focus on different parts 
of the image when generating each word. Instead of compressing the entire image into a single 
fixed vector, attention computes a *weighted sum* over all spatial patches the weights tell 
the model which patches are most relevant for the current word being generated.

For example, when generating the word *"dog"*, the model should attend to the patch containing 
the dog; when generating *"grass"*, it should shift focus to the ground patches. This is 
implemented here as **dot-product attention**: the LSTM hidden state at each timestep is 
compared (via dot product) against every image patch, producing a score. Softmax normalizes 
these scores into a probability distribution (the attention weights), which are then used to 
compute a weighted sum of the image patches the **context vector** passed to the decoder.


### Architecture Overview

```
Image patches (N_PATCHES, 2048)
    ↓ Dense(256, relu)              [trainable projection]
(N_PATCHES, 256) ←─────────────────────────────────────────────┐
                                                                │ dot-product attention
Caption tokens (max_len,)                                       │
    ↓ Embedding(vocab_size, 256)                                │
    ↓ LSTM(256, return_sequences=True)                          │
(max_len, 256) ── Dot([2,2]) ──→ scores (max_len, N_PATCHES)   │
                                  ↓ softmax                     │
                             weights (max_len, N_PATCHES)       │
                                  ↓ Dot([2,1]) with ───────────┘
                             context (max_len, 256)
                                  ↓ Concatenate with LSTM output
                             (max_len, 512)
                                  ↓ LSTM(256)
                                  ↓ Dropout(0.5)
                                  ↓ Dense(vocab_size, softmax)
                             next-word distribution (vocab_size,)
```

**Why `sparse_categorical_crossentropy`?**
Targets are integer word indices. Using `categorical_crossentropy` would require one-hot encoding every target into a `(batch, vocab_size)` matrix — roughly 800 MB for batch=256 and vocab≈3000. `sparse_categorical_crossentropy` accepts integer indices directly, avoiding that memory cost entirely.

### 3.1 Build Captioning Model with Attention (15 pts)

Complete `build_captioning_model(...)`. The architecture has two branches:

**Image branch:**
- Input shape: `(N_PATCHES, 2048)`
- A `Dense(embed_dim, activation='relu')` layer to project each patch to `embed_dim` dimensions.

**Text branch:**
- Input shape: `(max_len,)` (integer sequences)
- An `Embedding(vocab_size, embed_dim)` layer.
- An `LSTM(embed_dim, return_sequences=True)` layer.

**Attention:**
- Compute dot-product scores between LSTM outputs and projected image patches using `layers.Dot(axes=[2, 2])`.
- Apply softmax to get attention weights.
- Compute a weighted sum (context vector) over the image patches using `layers.Dot(axes=[2, 1])`.

**Decoder:**
- Concatenate the LSTM output and the context vector.
- Pass through a second `LSTM(lstm_units)` (this one does NOT return sequences).
- Apply `Dropout(0.5)`.
- A final `Dense(vocab_size, activation='softmax')` for word prediction.

**Compile** with Adam (lr=1e-4), `sparse_categorical_crossentropy` loss, and accuracy metric.

#### Expected Output (3.1)
```
Total params: 3,369,130 (12.85 MB)
Trainable params: 3,369,130 (12.85 MB)
Non-trainable params: 0 (0.00 B)
```
*(Total params should be approximately 3.3M–3.5M. All params should be trainable.)*


In [ ]:
def build_captioning_model(vocab_size, max_len, embed_dim=256, lstm_units=256):
    # ── Image branch ──
    img_input = layers.Input(shape=(N_PATCHES, 2048), name="image_input")
    # TODO: project image patches from 2048 to embed_dim with Dense + relu
    img_proj  = layers.Dense(embed_dim, activation='relu', name='img_projection')(img_input)

    # ── Text branch ──
    txt_input = layers.Input(shape=(max_len,), name="text_input")
    # TODO: Embedding layer, then LSTM with return_sequences=True
    txt_embed = layers.Embedding(vocab_size, embed_dim, mask_zero=False, name='txt_embedding')(txt_input)
    txt_lstm  = layers.LSTM(embed_dim, return_sequences=True, name='txt_lstm')(txt_embed)

    # ── Attention ──
    # TODO: dot-product scores → softmax weights → weighted context
    scores  = layers.Dot(axes=[2, 2], name='attention_scores')([txt_lstm, img_proj])
    weights = layers.Activation('softmax', name='attention_weights')(scores)
    context = layers.Dot(axes=[2, 1], name='attention_context')([weights, img_proj])

    # ── Decoder ──
    # TODO: Concatenate txt_lstm and context,
    #       then LSTM(lstm_units), Dropout(0.5), Dense(vocab_size, softmax)
    merged  = layers.Concatenate(name='decoder_concat')([txt_lstm, context])
    decoded = layers.LSTM(lstm_units, name='decoder_lstm')(merged)
    dropped = layers.Dropout(0.5, name='decoder_dropout')(decoded)
    output  = layers.Dense(vocab_size, activation='softmax', name='word_output')(dropped)

    model = keras.Model(inputs=[img_input, txt_input], outputs=output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_captioning_model(VOCAB_SIZE, MAX_LEN)
model.summary()


### 3.2 Data Generator + Train/Val/Test Split (10 pts)

Complete the `data_generator(...)` function, which implements **teacher forcing**.

**What is teacher forcing?** At training step `k`, we feed the *ground-truth* prefix `seq[:k]` as input and predict token `seq[k]` as the target. This is more stable than feeding the model's own (potentially wrong) predictions early in training. The trade-off is **exposure bias**: at inference there is no ground truth, so we must switch to greedy or beam decoding instead.

**Generator requirements:**
- For each caption sequence, generate one training example per step `k` in `range(1, len(seq))`.
- Input sequence: `pad_sequences([seq[:k]], maxlen=max_len, padding='post')[0]`
- Target word: `seq[k]`
- Yield `((np.array(X1, float32), np.array(X2, int32)), np.array(y, int32))` when the batch is full.
- The generator must loop **infinitely** (`while True`) — Keras will pull batches as needed.
- Shuffle the image list at the start of each pass.

The train/val/test split code (75% / 12.5% / 12.5%) is provided below.

**Expected output:**
```
Train: ~6069 | Val: ~1011 | Test: ~1011
```

In [ ]:
def data_generator(image_names, captions_seqs, features, word2idx,
                   max_len, vocab_size, batch_size=64):
    valid = [n for n in image_names if n in features and n in captions_seqs]
    while True:
        np.random.shuffle(valid)
        X1, X2, y = [], [], []
        for img_name in valid:
            img_feat = features[img_name]
            for seq in captions_seqs[img_name]:
                # TODO: for each step k from 1 to len(seq)-1:
                #   1. in_seq  = pad_sequences([seq[:k]], maxlen=max_len, padding='post')[0]
                #   2. out_word = seq[k]
                #   3. X1.append(img_feat), X2.append(in_seq), y.append(out_word)
                #   4. When len(X1) == batch_size, yield and reset X1/X2/y:
                #      yield ((np.array(X1, dtype=np.float32),
                #              np.array(X2, dtype=np.int32)),
                #             np.array(y, dtype=np.int32))

                # ---- YOUR CODE HERE ----
                for k in range(1, len(seq)):
                    in_seq = pad_sequences([seq[:k]], maxlen=max_len, padding='post')[0]
                    out_word = seq[k]
                    X1.append(img_feat)
                    X2.append(in_seq)
                    y.append(out_word)

                    if len(X1) == batch_size:
                        yield ((np.array(X1, dtype=np.float32),
                                np.array(X2, dtype=np.int32)),
                               np.array(y, dtype=np.int32))
                        X1, X2, y = [], [], []
                # ---- END YOUR CODE ----

        # yield any remaining samples (partial batch)
        if X1:
            yield ((np.array(X1, dtype=np.float32),
                    np.array(X2, dtype=np.int32)),
                   np.array(y, dtype=np.int32))

# TODO: Answer — what is teacher forcing? Why can't we use it at inference?
# Your answer: Teacher forcing means the decoder is trained with the true previous tokens
# as input at each step instead of its own predicted tokens. At inference, the true next
# words are unavailable, so the model must condition on the words it generated itself.
# That mismatch is why we switch to greedy or beam decoding at test time.

# ─── PROVIDED — train/val/test split ───
all_images = [img for img in captions_seqs.keys() if img in features]
np.random.seed(42)
np.random.shuffle(all_images)
n          = len(all_images)
train_imgs = all_images[:int(0.75 * n)]
val_imgs   = all_images[int(0.75 * n):int(0.875 * n)]
test_imgs  = all_images[int(0.875 * n):]
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")


---
## Part 4 — Training (15 pts)

Train the model using `model.fit(...)` with the data generators you built. You must:
1. Use **`ModelCheckpoint`** to save the best model (monitored on `val_loss`).
2. Use **`EarlyStopping`** with `patience=3` and `restore_best_weights=True`.
3. Pass `steps_per_epoch` and `validation_steps` (provided below).
4. Plot the training vs. validation loss curves.


#### Expected Output (Part 4 — Training)
```
Train samples: 357,708 | steps/epoch: 800
Val samples:   59,531   | val steps:   100

Epoch 1/20
800/800 ━━━━━━━━━━━━━━━━━━━━  ~249s  – accuracy: 0.1466 – loss: 5.4358 – val_accuracy: 0.1744 – val_loss: 5.0784
Epoch 2/20
...
```
*(Training loss should decrease to ~2.3–3.0. Val loss should reach ~3.0–3.5. EarlyStopping typically triggers around epoch 6–12.)*

In [ ]:
# ─── PROVIDED — step calculations ───
def count_samples(image_names, captions_seqs):
    total = 0
    for img in image_names:
        if img in captions_seqs:
            for seq in captions_seqs[img]:
                total += max(0, len(seq) - 1)
    return total

BATCH_SIZE       = 256
MAX_STEPS        = 800
MAX_VAL_STEPS    = 100

train_samples    = count_samples(train_imgs, captions_seqs)
val_samples      = count_samples(val_imgs,   captions_seqs)
steps_per_epoch  = min(train_samples // BATCH_SIZE, MAX_STEPS)
validation_steps = min(max(1, val_samples // BATCH_SIZE), MAX_VAL_STEPS)
print(f"Train samples: {train_samples:,} | steps/epoch: {steps_per_epoch}")
print(f"Val samples:   {val_samples:,}   | val steps:   {validation_steps}")


In [ ]:
tf.keras.backend.clear_session()
model = build_captioning_model(VOCAB_SIZE, MAX_LEN)

# TODO: create a callbacks list with:
#   1. ModelCheckpoint — save "best_model.keras", monitor val_loss, save_best_only=True, verbose=1
#   2. EarlyStopping — monitor val_loss, patience=3, restore_best_weights=True, verbose=1

# ---- YOUR CODE HERE ----
callbacks = [
    ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1)
]
# ---- END YOUR CODE ----

train_gen = data_generator(train_imgs, captions_seqs, features,
                            word2idx, MAX_LEN, VOCAB_SIZE, BATCH_SIZE)
val_gen   = data_generator(val_imgs, captions_seqs, features,
                            word2idx, MAX_LEN, VOCAB_SIZE, BATCH_SIZE)

# TODO: call model.fit with train_gen, val_gen, steps_per_epoch,
#       validation_steps, epochs=20, and your callbacks

# ---- YOUR CODE HERE ----
history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    epochs=20,
    callbacks=callbacks
)
# ---- END YOUR CODE ----

# Expected: EarlyStopping fires around epoch 8-12
# Expected: final train loss 2.3–2.8 | val loss 3.0–3.4


### 4.1 Loss Curves

Run the provided cell below to plot your training and validation loss curves after training completes.

In [ ]:
# PROVIDED — run after training completes
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()
# Expected: val_loss plateaus and nudges up while train_loss keeps falling — mild overfitting.

### 4.2 Reflection

**Q1:** Describe the shape of the loss curves. Is there evidence of overfitting? How can you tell?

**Q2:** Explain what `EarlyStopping` with `restore_best_weights=True` does. Why does `restore_best_weights` matter?

*Write your answers below (double-click to edit):*


**Q1 answer:** The training loss should steadily decrease, while validation loss usually improves at first and then flattens or starts creeping upward. That gap suggests mild overfitting because the model continues fitting the training set better than unseen validation data.

**Q2 answer:** `EarlyStopping` stops training once `val_loss` has not improved for the chosen patience window. With `restore_best_weights=True`, the model parameters are rolled back to the epoch that achieved the lowest validation loss instead of keeping the overfit weights from the final epoch.


---
## Part 5 — Caption Generation / Inference (15 pts)

Implement **greedy decoding**: start with `<start>`, repeatedly predict the most probable next word, and stop when `<end>` is generated or `max_len` steps have been taken.

**Inference vs training:** During training we fed the *ground-truth* prefix (teacher forcing). At inference there is no ground truth, so at each step we feed the model's **own previous prediction** and keep extending the sequence.

**Requirements:**
- Start: `in_seq = [word2idx['<start>']]`
- At each step: pad `in_seq` to `max_len` with `pad_sequences([in_seq], maxlen=max_len, padding='post')`.
- Reshape `image_feature` to `(1, N_PATCHES, 2048)` before calling `model.predict`.
- Take `np.argmax(preds[0])` to get the next token index.
- Stop if the predicted token is `<end>` or `<pad>`.
- Return a **string** — exclude `<start>`, `<end>`, `<pad>`, and `<unk>` from the output.

In [ ]:
def generate_caption(model, image_feature, word2idx, idx2word, max_len):
    # TODO:
    #   1. Start with in_seq = [word2idx['<start>']]
    #   2. Loop up to max_len times:
    #      a. Pad in_seq to max_len (post-padding)
    #      b. Reshape image_feature to (1, N_PATCHES, 2048)
    #      c. model.predict([img_feat, padded])
    #      d. Take argmax of predictions → next word index
    #      e. If next word is <end> or <pad>, stop
    #      f. Otherwise append to in_seq
    #   3. Convert indices to words (skip <start>, <unk>)
    #   4. Return the caption as a string

    # ---- YOUR CODE HERE ----
    in_seq = [word2idx['<start>']]
    img_feat = image_feature.reshape(1, N_PATCHES, 2048)

    for _ in range(max_len):
        padded = pad_sequences([in_seq], maxlen=max_len, padding='post')
        preds = model.predict([img_feat, padded], verbose=0)
        next_idx = int(np.argmax(preds[0]))

        if next_idx in (word2idx['<end>'], word2idx['<pad>']):
            break
        in_seq.append(next_idx)

    words = []
    for idx in in_seq:
        word = idx2word.get(idx, '<unk>')
        if word in ('<start>', '<unk>'):
            continue
        if word in ('<end>', '<pad>'):
            break
        words.append(word)

    return ' '.join(words)
    # ---- END YOUR CODE ----

# ─── PROVIDED — display helper ───
def show_caption(img_name):
    caption = generate_caption(model, features[img_name], word2idx, idx2word, MAX_LEN)
    img     = Image.open(os.path.join(IMAGES_DIR, img_name))
    plt.figure(figsize=(6, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(caption, fontsize=11, wrap=True)
    plt.tight_layout()
    plt.show()
    print("Generated :", caption)
    print("References:")
    for ref in captions_clean.get(img_name, [])[:3]:
        print(" ", ref)
    print()

# Show captions for 5 test images
for img_name in test_imgs[:5]:
    show_caption(img_name)


---
## Part 6 — BLEU Evaluation (10 pts)

**BLEU (Bilingual Evaluation Understudy)** measures n-gram overlap between a generated caption and multiple reference captions.

- **BLEU-1**: unigram precision — `weights=(1, 0, 0, 0)`
- **BLEU-2**: geometric mean of unigram + bigram precision — `weights=(0.5, 0.5, 0, 0)`
- **`SmoothingFunction().method1`**: adds a small pseudo-count to avoid zero scores when a higher-order n-gram has no matches.

**Exact format required by `corpus_bleu`:**
- `references`: `[[ref1_tokens, ref2_tokens, ...], ...]` — a list of lists of lists
- `hypotheses`: `[[word1, word2, ...], ...]` — a list of lists (one per image)

Both must be **tokenized** (lists of strings), not raw caption strings.

#### Expected Output (6.0 — BLEU Evaluation)
```
BLEU eval: 100%|██████████| 500/500
BLEU-1: 0.4938
BLEU-2: 0.3258
```
*(Expect BLEU-1 in the range 0.50–0.62 and BLEU-2 in the range 0.28–0.38. Values may vary slightly between runs.)*

In [ ]:
def evaluate_bleu(model, test_images, max_eval=500):
    references, hypotheses = [], []
    smoother = SmoothingFunction().method1
    eval_imgs = test_images[:max_eval]

    for img_name in tqdm(eval_imgs, desc="BLEU eval"):
        if img_name not in features or img_name not in captions_clean:
            continue

        # TODO:
        #   1. Generate a caption for this image
        #   2. Append the tokenized caption (list of words) to hypotheses
        #   3. Append the list of tokenized reference captions to references

        # ---- YOUR CODE HERE ----
        pred_caption = generate_caption(model, features[img_name], word2idx, idx2word, MAX_LEN)
        pred_tokens = pred_caption.split()
        ref_tokens = [ref.split() for ref in captions_clean[img_name]]

        hypotheses.append(pred_tokens)
        references.append(ref_tokens)
        # ---- END YOUR CODE ----

    # TODO: compute BLEU-1 and BLEU-2 using corpus_bleu with the correct weights

    # ---- YOUR CODE HERE ----
    bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoother)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoother)
    # ---- END YOUR CODE ----
    return bleu1, bleu2

bleu1, bleu2 = evaluate_bleu(model, test_imgs)
print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")


### 6.1 Qualitative Analysis

Find and display:
- **2 good examples** : images where the generated caption has **≥ 5 words** in common with the reference word set.
- **2 bad examples** : images where the generated caption has **≤ 2 words** in common with the reference word set.

Use `show_caption(img_name)` for each. You can programmatically find examples by comparing `set(caption.split())` against `set(' '.join(refs).split())`, or browse outputs manually.

After displaying the examples, write a **3–5 sentence reflection** in the cell below explaining what kinds of scenes the model handles well vs. poorly.

In [ ]:
# TODO: display 2 good captions (>=5 word overlap) and 2 bad captions (<=2 word overlap)
#       Use show_caption(img_name) to display each one.
#       Hint: overlap = len(set(caption.split()) & set(' '.join(refs).split()))

good_examples, bad_examples = [], []

for img_name in test_imgs:
    caption = generate_caption(model, features[img_name], word2idx, idx2word, MAX_LEN)
    refs = captions_clean.get(img_name, [])
    overlap = len(set(caption.split()) & set(' '.join(refs).split()))

    if overlap >= 5 and len(good_examples) < 2:
        good_examples.append(img_name)
    if overlap <= 2 and len(bad_examples) < 2:
        bad_examples.append(img_name)

    if len(good_examples) == 2 and len(bad_examples) == 2:
        break

# ---- GOOD CAPTIONS ----
for img_name in good_examples:
    show_caption(img_name)

# ---- BAD CAPTIONS ----
for img_name in bad_examples:
    show_caption(img_name)


### 6.2 Reflection

**Q1:** What kinds of images or scenes does the model caption well? Why?

**Q2:** What does it struggle with? Name at least two failure modes.

**Q3:** What architectural change could improve performance? (1–2 sentences.)

*Write your answers below:*


**Q1 answer:** The model usually does better on common Flickr8k scenes with strong visual patterns, such as dogs running, children playing, or people outdoors, because those objects and actions appear often in the training data and have clearer visual cues.

**Q2 answer:** It struggles with fine-grained details and less frequent concepts. Two common failure modes are confusing similar objects/actions (for example, running vs. jumping) and generating generic captions that miss important specifics such as counts, attributes, or relationships.

**Q3 answer:** A stronger decoder with a better attention mechanism could improve performance, such as adding Bahdanau-style attention or replacing the LSTM decoder with a Transformer-based text decoder. More regularization and a stronger visual encoder can also help.


---
## Bonus - Beam Search (+10 pts)

Implement **beam search** decoding as an alternative to greedy decoding. Instead of always picking the single most likely next word, beam search maintains the top `beam_width` candidate sequences at each step.

**Algorithm:**
1. Start: `beams = [([word2idx['<start>']], 0.0)]` - one sequence with cumulative log prob = 0.
2. Each step: expand every beam by its top-`k` next tokens → `beam_width × beam_width` candidates.
3. Keep the top `beam_width` candidates by cumulative log score.
4. When a beam generates `<end>`, move it to a `completed` list.
5. After `max_len` steps, return the highest-scoring sequence from `completed + remaining beams`.

>  **Use log probabilities**, not raw probabilities. Multiplying many small probabilities across steps underflows to 0.0 in float32. Use `math.log(prob + 1e-10)` and accumulate by addition.

**Requirements:**
- `beam_width` must be ≥ 2 (beam_width=1 is equivalent to greedy — no benefit).
- Implement `evaluate_bleu_beam` and compare BLEU scores against greedy.

#### Expected Output (Bonus — Beam Search)
```
Beam(k=3): 100%|██████████| 200/200
Greedy  — BLEU-1: 0.4938 | BLEU-2: 0.3258
Beam(3) — BLEU-1: 0.5804 | BLEU-2: 0.3862
```
*(Beam search should improve over greedy. Expect BLEU-1 beam in the range 0.53–0.65.)*

In [ ]:
def beam_search_caption(model, image_feature, word2idx, idx2word, max_len, beam_width=3):
    # TODO: implement beam search
    #   1. Start with a single beam: ([<start> index], 0.0 log prob)
    #   2. At each step, expand each beam by beam_width candidates
    #   3. Keep the top beam_width beams by cumulative log probability
    #   4. Stop beams that generate <end>
    #   5. Return the best completed sequence

    # ---- YOUR CODE HERE ----
    start_idx = word2idx['<start>']
    end_idx = word2idx['<end>']
    pad_idx = word2idx['<pad>']

    beams = [([start_idx], 0.0)]
    completed = []
    img_feat = image_feature.reshape(1, N_PATCHES, 2048)

    for _ in range(max_len):
        candidates = []

        for seq, score in beams:
            last_token = seq[-1]
            if last_token == end_idx:
                completed.append((seq, score))
                candidates.append((seq, score))
                continue

            padded = pad_sequences([seq], maxlen=max_len, padding='post')
            preds = model.predict([img_feat, padded], verbose=0)[0]

            top_indices = np.argsort(preds)[-beam_width:][::-1]
            for idx in top_indices:
                prob = float(preds[idx])
                prob = max(prob, 1e-10)
                candidates.append((seq + [int(idx)], score + math.log(prob)))

        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]

        if all(seq[-1] in (end_idx, pad_idx) for seq, _ in beams):
            completed.extend(beams)
            break

    best_seq = max(completed if completed else beams, key=lambda x: x[1])[0]

    words = []
    for idx in best_seq:
        word = idx2word.get(idx, '<unk>')
        if word in ('<start>', '<unk>'):
            continue
        if word in ('<end>', '<pad>'):
            break
        words.append(word)

    return ' '.join(words)
    # ---- END YOUR CODE ----

# Test on a few images
for img_name in test_imgs[:3]:
    greedy = generate_caption(model, features[img_name], word2idx, idx2word, MAX_LEN)
    beam   = beam_search_caption(model, features[img_name], word2idx, idx2word, MAX_LEN, beam_width=3)
    print(f"Greedy: {greedy}")
    print(f"Beam:   {beam}")
    print()


In [ ]:
def evaluate_bleu_beam(model, test_images, beam_width=3, max_eval=200):
    # TODO: same structure as evaluate_bleu but call beam_search_caption instead
    #       references format: [[ref1_tokens, ref2_tokens, ...], ...]
    #       hypotheses format: [[word1, word2, ...], ...]

    # ---- YOUR CODE HERE ----
    references, hypotheses = [], []
    smoother = SmoothingFunction().method1
    eval_imgs = test_images[:max_eval]

    for img_name in tqdm(eval_imgs, desc="Beam BLEU eval"):
        if img_name not in features or img_name not in captions_clean:
            continue

        pred_caption = beam_search_caption(model, features[img_name], word2idx, idx2word, MAX_LEN, beam_width=beam_width)
        hypotheses.append(pred_caption.split())
        references.append([ref.split() for ref in captions_clean[img_name]])

    bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoother)
    bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoother)
    return bleu1, bleu2
    # ---- END YOUR CODE ----


# Compare greedy vs beam
beam_b1, beam_b2 = evaluate_bleu_beam(model, test_imgs, beam_width=3)
print(f"Greedy  — BLEU-1: {bleu1:.4f} | BLEU-2: {bleu2:.4f}")
print(f"Beam(3) — BLEU-1: {beam_b1:.4f} | BLEU-2: {beam_b2:.4f}")
# Expected: Beam BLEU-1 improves by ~0.03-0.05 over greedy


---
## Expected Output Ranges

Use this table to sanity-check your results at each step.

| Metric | Expected |
|--------|----------|
| `Total images` | 8092 |
| `VOCAB_SIZE` | 2,900 – 3,100 |
| `MAX_LEN` | 35 – 40 |
| Special token indices | `<pad>=0, <start>=1, <end>=2, <unk>=3` |
| Feature shape per image | `(64, 2048)` or `(49, 2048)` depending on TF version |
| InceptionV3 trainable weights | 0 |
| EarlyStopping best epoch | 6 – 12 |
| Final train loss | 2.3 – 3.0 |
| Final val loss | 3.0 – 3.5 |
| BLEU-1 greedy | 0.50 – 0.62 |
| BLEU-2 greedy | 0.28 – 0.38 |
| BLEU-1 beam=3 | 0.53 – 0.65 |

**If your BLEU is below 0.40:**
1. Check feature shape — must be `(N_PATCHES, 2048)`, not `(2048,)`. Delete `features_attention.pkl` and re-run Part 2 if wrong.
2. Verify `pooling=None` in `build_feature_extractor`.
3. Verify learning rate is `1e-4`, not `1e-3`.